In [10]:
import torch
import time
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
import math
from time import perf_counter
import json
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

from quantum_Evaluation import evaluate_predictions, analyze_vqc_angles, save_model_and_artifacts

CURRENT_SEED = 17
COMPONENTS   = [32, 16, 8, 4]
N_QUBITS = 8

os.makedirs('../compressedFeatures', exist_ok=True)

In [11]:
def angular_scaling(X_train, X_val, X_test):
    scaler = MinMaxScaler(feature_range=(-np.pi/2, np.pi/2))
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    X_val_scaled  = np.clip(X_val_scaled, -np.pi/2, np.pi/2)
    X_test_scaled = np.clip(X_test_scaled, -np.pi/2, np.pi/2)

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

# Pre-elaborazione e salvataggio dataset
for d in COMPONENTS:
    path = f'../compressedFeatures/pca_d{d}_seed_{CURRENT_SEED}.pt'
    if os.path.exists(path):
        ckpt = torch.load(path, weights_only=False)
        X_train = ckpt['train'].numpy()
        X_val   = ckpt['val'].numpy()
        X_test  = ckpt['test'].numpy()

        X_train_ang, X_val_ang, X_test_ang, scaler = angular_scaling(X_train, X_val, X_test)

        torch.save({
            'train_ang':       torch.from_numpy(X_train_ang).float(),
            'val_ang':         torch.from_numpy(X_val_ang).float(),
            'test_ang':        torch.from_numpy(X_test_ang).float(),
            'y_train':         ckpt['y_train'],
            'y_val':           ckpt['y_val'],
            'y_test':          ckpt['y_test'],
            'd':               d,
            'seed':            CURRENT_SEED,
            'angular_scaling': 'minmax_(0,2pi]'
        }, f'../compressedFeatures/angular_d{d}_seed_{CURRENT_SEED}.pt')

def build_encoding_layer(d, n_qubits, block_idx):
    params = ParameterVector(f'x_{block_idx}', n_qubits) 
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(params[i], i)
    return qc, params

def build_ansatz_layer(n_qubits, reps, entanglement, block_idx):
    ansatz = EfficientSU2(
        num_qubits=n_qubits,
        reps=reps,
        entanglement=entanglement,
        parameter_prefix=f'θ_{block_idx}'
    )
    return ansatz.decompose()

def build_vqc_circuit(d, n_qubits=N_QUBITS, reps=1, entanglement='linear'):
    n_blocks = math.ceil(d / n_qubits)
    qc = QuantumCircuit(n_qubits)
    all_input_params       = []
    all_variational_params = []

    for block in range(n_blocks):
        enc_layer, enc_params = build_encoding_layer(d, n_qubits, block)
        qc.compose(enc_layer, inplace=True)
        all_input_params.extend(enc_params)
        
        ansatz = build_ansatz_layer(n_qubits, reps, entanglement, block)
        qc.compose(ansatz, inplace=True)
        all_variational_params.extend(ansatz.parameters)

    return qc, all_input_params, all_variational_params, n_blocks

In [12]:
def single_pauli_observables(n_qubits, op):
    observables = []
    for qubit in range(n_qubits):
        pauli = ["I"] * n_qubits
        pauli[n_qubits - 1 - qubit] = op
        observables.append(SparsePauliOp.from_list([("".join(pauli), 1.0)]))
    return observables

def pair_pauli_observables(n_qubits, op, pairs):
    observables = []
    for (i, j) in pairs:
        pauli = ["I"] * n_qubits
        pauli[n_qubits - 1 - i] = op
        pauli[n_qubits - 1 - j] = op
        observables.append(SparsePauliOp.from_list([("".join(pauli), 1.0)]))
    return observables

def build_readout_observables(n_qubits, readout="z"):
    singles_xyz = (single_pauli_observables(n_qubits, "X")
                 + single_pauli_observables(n_qubits, "Y")
                 + single_pauli_observables(n_qubits, "Z"))

    if readout == "z":
        return single_pauli_observables(n_qubits, "Z")
    elif readout == "x_z":
        return (single_pauli_observables(n_qubits, "X")
              + single_pauli_observables(n_qubits, "Z"))
    elif readout == "x_y_z":
        return singles_xyz
    elif readout == "x_y_z_pair_nn":
        nn_pairs = [(i, i+1) for i in range(n_qubits - 1)]
        pair_obs = []
        for op in ["X", "Y", "Z"]:
            pair_obs += pair_pauli_observables(n_qubits, op, nn_pairs)
        return singles_xyz + pair_obs          
    elif readout == "x_y_z_pair_all":
        all_pairs = [(i, j) for i in range(n_qubits) for j in range(i+1, n_qubits)]
        pair_obs = []
        for op in ["X", "Y", "Z"]:
            pair_obs += pair_pauli_observables(n_qubits, op, all_pairs)
        return singles_xyz + pair_obs          
    else:
        raise ValueError(f"Readout non supportato: {readout}")

In [13]:
class _BatchedEstimatorPauliSPSAFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, model: "VQC", x: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
        features = model._estimate_readout_features(
            x.detach().cpu().numpy().astype(np.float64),
            weights.detach().cpu().numpy().astype(np.float64),
        )
        ctx.model = model
        ctx.save_for_backward(x.detach(), weights.detach())
        return torch.tensor(features, dtype=x.dtype, device=x.device)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        x, weights = ctx.saved_tensors
        model: VQC = ctx.model
        grad_weights = model._spsa_weight_gradient(
            x.detach().cpu().numpy().astype(np.float64),
            weights.detach().cpu().numpy().astype(np.float64),
            grad_output.detach().cpu().numpy().astype(np.float64),
        )
        return None, None, torch.tensor(grad_weights, dtype=weights.dtype, device=weights.device)

In [14]:
class VQC(nn.Module):
    def __init__(self, n_qubits: int, quantum_circuit: QuantumCircuit, obs: list, gradient_mode='SPSA', input_params=None, weight_params=None, target_classes=2, **kwargs):
        super(VQC, self).__init__()
        self._use_batched_estimator_spsa = (gradient_mode == 'estimator_pauli_batched_spsa')
        self.n_qubits = n_qubits
        self.quantum_circuit = quantum_circuit
        self.input_params = input_params
        self.weight_params = weight_params

        estimator = AerEstimator()
        simulator_options = {"method": "statevector"}
        if kwargs.get('use_gpu', True):
            simulator_options.update({
                "device": "GPU",          
                "cuStateVec_enable": True 
            })
        estimator.options.simulator = simulator_options
        
        self.q_weights = nn.Parameter(torch.empty(len(list(self.weight_params))).uniform_(-0.01, 0.01))
        num_observables = len(obs)
        self.head_classical_linear_layer = nn.Linear(num_observables, target_classes)  

        if self._use_batched_estimator_spsa:
            self.estimator = estimator
            self.observables = [[observable] for observable in obs]
            self.readout_dim = num_observables
            self.estimator_precision = float(kwargs.get('estimator_precision', 0.0))
            self.spsa_epsilon = float(kwargs.get('spsa_epsilon', 1e-6))
            self.spsa_batch_size = int(kwargs.get('spsa_batch_size', 1))
            if self.spsa_epsilon <= 0: raise ValueError(f"spsa_epsilon deve essere > 0")
            if self.spsa_batch_size <= 0: raise ValueError(f"spsa_batch_size deve essere > 0")
            self._spsa_rng = np.random.default_rng(int(kwargs.get('seed', 0)))

            source_params = list(self.input_params) + list(self.weight_params)
            source_index = {param: index for index, param in enumerate(source_params)}
            self._parameter_order = list(self.quantum_circuit.parameters)
            try:
                self._parameter_source_indices = np.array(
                    [source_index[param] for param in self._parameter_order], dtype=np.int64,
                )
            except KeyError as exc:
                raise RuntimeError("L'ordine dei parametri del circuito contiene parametri non tracciati.") from exc
            return

    def forward(self, x):    
        if self._use_batched_estimator_spsa:
            q_out = _BatchedEstimatorPauliSPSAFunction.apply(self, x, self.q_weights) 
        else:
            q_out = self.quantum_layer(x)
        logits = self.head_classical_linear_layer(q_out)
        return logits
    
    def _ordered_parameter_values(self, input_values: np.ndarray, weights: np.ndarray) -> np.ndarray:
        if input_values.ndim == 1: input_values = input_values.reshape(1, -1)
        if weights.ndim == 1: weight_values = np.broadcast_to(weights, (input_values.shape[0], weights.shape[0]))
        else: weight_values = weights
        source_values = np.concatenate([input_values, weight_values], axis=1)
        return source_values[:, self._parameter_source_indices]

    def _estimate_readout_features(self, input_values: np.ndarray, weights: np.ndarray) -> np.ndarray:
        parameter_values = self._ordered_parameter_values(input_values, weights)
        pub = (self.quantum_circuit, self.observables, parameter_values)
        result = self.estimator.run([pub], precision=self.estimator_precision).result()
        evs = np.asarray(result[0].data.evs, dtype=np.float64)
        expected_size = self.readout_dim * input_values.shape[0]
        if evs.size != expected_size:
            raise RuntimeError(f"EstimatorV2 ha restituito evs size={evs.size}; atteso {expected_size}")
        return evs.reshape(self.readout_dim, input_values.shape[0]).T

    def _spsa_weight_gradient(self, input_values: np.ndarray, weights: np.ndarray, upstream_gradient: np.ndarray) -> np.ndarray:
        deltas = self._spsa_rng.choice(np.array([-1.0, 1.0], dtype=np.float64), size=(self.spsa_batch_size, weights.shape[0]))
        perturbed = []
        for delta in deltas:
            perturbed.append(weights + self.spsa_epsilon * delta)
            perturbed.append(weights - self.spsa_epsilon * delta)
        perturbed_weights = np.repeat(np.stack(perturbed, axis=0), input_values.shape[0], axis=0)
        repeated_inputs = np.tile(input_values, (len(perturbed), 1))
        features = self._estimate_readout_features(repeated_inputs, perturbed_weights)
        features = features.reshape(len(perturbed), input_values.shape[0], -1)

        grad = np.zeros_like(weights, dtype=np.float64)
        for index, delta in enumerate(deltas):
            plus = features[2 * index]
            minus = features[2 * index + 1]
            directional = np.sum(upstream_gradient * (plus - minus)) / (2.0 * self.spsa_epsilon)
            grad += directional * delta
        return grad / float(self.spsa_batch_size)

In [15]:
def build_hybrid_model(d, n_qubits=N_QUBITS, reps=1, entanglement='linear', readout="z", seed=CURRENT_SEED, gradient_mode='estimator_pauli_batched_spsa', **kwargs):
    qc, input_params, var_params, n_blocks = build_vqc_circuit(d, n_qubits, reps, entanglement)
    observables = build_readout_observables(n_qubits, readout)

    model = VQC(
        n_qubits=n_qubits, quantum_circuit=qc, obs=observables,
        gradient_mode=gradient_mode, input_params=input_params,
        weight_params=var_params, target_classes=4, seed=seed, **kwargs
    )
   
    return model

def spsa_step(model, X_batch, y_batch, loss_fn, step, a0=0.1, alpha=0.602, spsa_epsilon=0.1, spsa_batch_size=1, classic_optimizer=None):
    # Usato SOLO come fallback se gradient_mode != 'estimator_pauli_batched_spsa'
    ak = a0 / (step + 1) ** alpha
    quantum_params = list(model[0].parameters())
    originals = [p.data.clone() for p in quantum_params]
    grad_accum = [torch.zeros_like(p) for p in quantum_params]
    losses = []

    with torch.no_grad():
        all_deltas = []
        for _ in range(spsa_batch_size):
            deltas = [torch.randint(0, 2, p.shape).float() * 2 - 1 for p in quantum_params]
            all_deltas.append(deltas)

        for deltas in all_deltas:
            for p, orig, d in zip(quantum_params, originals, deltas): p.data = orig + spsa_epsilon * d
            loss_plus = loss_fn(model(X_batch), y_batch)

            for p, orig, d in zip(quantum_params, originals, deltas): p.data = orig - spsa_epsilon * d
            loss_minus = loss_fn(model(X_batch), y_batch)

            grad_scalar = (loss_plus - loss_minus) / (2 * spsa_epsilon)
            for g, d in zip(grad_accum, deltas): g += grad_scalar * d
            losses.append(((loss_plus + loss_minus) / 2).item())

        for p, orig, g in zip(quantum_params, originals, grad_accum):
            p.data = orig - ak * (g / spsa_batch_size)

    if classic_optimizer is not None:
        with torch.no_grad(): quantum_out = model[0](X_batch)
        logits = model[1](quantum_out.detach())
        loss_classic = loss_fn(logits, y_batch)
        classic_optimizer.zero_grad()
        loss_classic.backward()
        classic_optimizer.step()

    return float(np.mean(losses))

In [16]:
def apply_padding(X, d, n_qubits):
    n_blocks = math.ceil(d / n_qubits)
    padded_size = n_blocks * n_qubits
    if padded_size == d: return X  
    pad = torch.zeros(X.shape[0], padded_size - d)
    return torch.cat([X, pad], dim=1)

In [17]:
def train_vqc(d, n_qubits=N_QUBITS, n_epochs=50, batch_size=32, patience=10, seed=CURRENT_SEED,
              a0=1.0, spsa_epsilon=1e-3, spsa_batch_size=1, readout="z", reps=1, gradient_mode='estimator_pauli_batched_spsa'):
    
    ckpt = torch.load(f'../compressedFeatures/angular_d{d}_seed_{seed}.pt', weights_only=False)
    X_train = apply_padding(ckpt['train_ang'], d, n_qubits)
    X_val   = apply_padding(ckpt['val_ang'],   d, n_qubits)
    X_test  = apply_padding(ckpt['test_ang'],  d, n_qubits)
    y_train = ckpt['y_train'].squeeze().long()
    y_val   = ckpt['y_val'].squeeze().long()
    y_test  = ckpt['y_test'].squeeze().long()

    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # Costruzione Modello
    model = build_hybrid_model(
        d, n_qubits, reps=reps, readout=readout, seed=seed, 
        gradient_mode=gradient_mode, spsa_epsilon=spsa_epsilon, spsa_batch_size=spsa_batch_size
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Benchmark veloce
    X_bench, y_bench = X_train[:batch_size], y_train[:batch_size]
    t0 = time.perf_counter()
    optimizer.zero_grad()
    _ = loss_fn(model(X_bench), y_bench).backward()
    elapsed = time.perf_counter() - t0
    n_batches = math.ceil(len(X_train) / batch_size)
    print(f"benchmark: {elapsed:.2f}s/batch | stima epoca: {elapsed * n_batches:.1f}s")
    print('-' * 50)

    best_val_loss = float('inf')
    patience_counter = 0
    best_weights = None
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(n_epochs):
        model.train()
        epoch_losses, epoch_times = [], []

        global_step = 0
        
        for X_batch, y_batch in train_loader:
            t_b0 = time.time()
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            loss_val = loss.item()
                         
            epoch_losses.append(loss_val)
            epoch_times.append(time.time() - t_b0)
            global_step += 1

        train_loss = np.mean(epoch_losses)
        
        # Validation evaluation
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        print(f'epoch {epoch+1:3d}/{n_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | tempo={np.sum(epoch_times):.1f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping a epoch {epoch+1}')
                break

    model.load_state_dict(best_weights)
    return model, history

In [21]:
# =====================================================================
#  ESPERIMENTI pdf prof
# =====================================================================
EXPERIMENTS = [
    dict(d=4,  readout="z",             reps=1, spsa_batch_size=1, n_epochs=50, a0=0.5),
    dict(d=8,  readout="z",             reps=1, spsa_batch_size=1, n_epochs=50, a0=0.5),
    dict(d=16, readout="x_y_z",         reps=1, spsa_batch_size=1, n_epochs=50, a0=0.5),
    dict(d=16, readout="x_y_z_pair_nn", reps=1, spsa_batch_size=1, n_epochs=50, a0=0.5),
    dict(d=32, readout="x_y_z_pair_nn", reps=2, spsa_batch_size = 2, n_epochs=100, a0=0.5),
]

os.makedirs('../artifacts', exist_ok=True)
results_summary = []

for exp in EXPERIMENTS:
    d         = exp['d']
    readout   = exp['readout']
    reps      = exp['reps']
    spsa_bs   = exp['spsa_batch_size']
    tag       = f"d{d}_{readout}_reps{reps}"
    n_epochs  = exp.get('n_epochs', 50)

    # Nota: lo script ora salva il file come 'vqc_metrics_...', modifichiamo il controllo esistenza di conseguenza
    artifact_path = f'../artifacts/metrics/vqc_metrics_{tag}_seed_{CURRENT_SEED}.json'
    if os.path.exists(artifact_path):
        print(f"[SKIP] {tag} — artifact già presente\n")
        continue

    print(f"\n{'='*60}\nESPERIMENTO: {tag}\n{'='*60}")
    t_start = perf_counter()

    try:
        # 1. Addestramento del modello
        model, history = train_vqc(
            d=d, n_qubits=N_QUBITS, n_epochs=n_epochs, 
            batch_size=32, patience=15, seed=CURRENT_SEED, spsa_epsilon=0.1, spsa_batch_size=spsa_bs,
            readout=readout, reps=reps
        )

        # Caricamento dati di validazione e test per la valutazione finale
        ckpt = torch.load(f'../compressedFeatures/angular_d{d}_seed_{CURRENT_SEED}.pt', weights_only=False)
        X_val  = apply_padding(ckpt['val_ang'],  d, N_QUBITS)
        X_test = apply_padding(ckpt['test_ang'], d, N_QUBITS)
        y_val  = ckpt['y_val'].squeeze().long()
        y_test = ckpt['y_test'].squeeze().long()

        # 2. VALUTAZIONE TRAMITE SCRIPT
        val_metrics,  val_preds,  val_probs  = evaluate_predictions(model, X_val,  y_val)
        test_metrics, test_preds, test_probs = evaluate_predictions(model, X_test, y_test)

        # 3. ANALISI DEGLI ANGOLI TRAMITE SCRIPT
        print(f"\n--- ANALISI DEGLI ANGOLI APPRESI ({tag}) ---")
        optimal_angles = analyze_vqc_angles(model, d, N_QUBITS)

        elapsed_min = (perf_counter() - t_start) / 60
        
        
        # 4. SALVATAGGIO DEI MODELLI E ARTIFACTS TRAMITE SCRIPT
        val_data = (y_val, val_preds, val_probs)
        test_data = (y_test, test_preds, test_probs)
        
        save_model_and_artifacts(
            model=model,
            val_metrics=val_metrics,
            test_metrics=test_metrics,
            val_data=val_data,
            test_data=test_data,
            history=history,
            d=d,
            n_qubits=N_QUBITS,
            tag=tag,
            seed=CURRENT_SEED,
            elapsed_min=elapsed_min,
            readout=readout,
            reps=reps,
            spsa_bs=spsa_bs,
            base_path='../artifacts'
        )

        results_summary.append({
            'tag': tag, 'val_f1': val_metrics['macro_f1'], 'test_f1': test_metrics['macro_f1'],
            'val_auroc': val_metrics['macro_auroc'], 'elapsed_min': round(elapsed_min, 2), 'status': 'OK'
        })
        
    except Exception as e:
        print(f"[ERRORE] {tag}: {e}")
        results_summary.append({'tag': tag, 'status': f'ERRORE: {e}'})
        continue

[SKIP] d4_z_reps1 — artifact già presente

[SKIP] d8_z_reps1 — artifact già presente

[SKIP] d16_x_y_z_reps1 — artifact già presente

[SKIP] d16_x_y_z_pair_nn_reps1 — artifact già presente


ESPERIMENTO: d32_x_y_z_pair_nn_reps2


C:\Users\shali\AppData\Local\Temp\ipykernel_26908\1455625848.py:43: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(


benchmark: 0.28s/batch | stima epoca: 34.6s
--------------------------------------------------
epoch   1/100 | train_loss=1.2760 | val_loss=1.1750 | tempo=34.1s
epoch   2/100 | train_loss=1.1040 | val_loss=1.0682 | tempo=36.7s
epoch   3/100 | train_loss=1.0219 | val_loss=1.0155 | tempo=31.2s
epoch   4/100 | train_loss=0.9773 | val_loss=0.9751 | tempo=30.6s
epoch   5/100 | train_loss=0.9515 | val_loss=0.9583 | tempo=29.6s
epoch   6/100 | train_loss=0.9375 | val_loss=0.9506 | tempo=29.3s
epoch   7/100 | train_loss=0.9258 | val_loss=0.9563 | tempo=29.5s
epoch   8/100 | train_loss=0.9250 | val_loss=0.9548 | tempo=29.7s
epoch   9/100 | train_loss=0.9212 | val_loss=0.9405 | tempo=29.5s
epoch  10/100 | train_loss=0.9156 | val_loss=0.9321 | tempo=29.6s
epoch  11/100 | train_loss=0.9201 | val_loss=0.9432 | tempo=29.5s
epoch  12/100 | train_loss=0.9173 | val_loss=0.9329 | tempo=29.7s
epoch  13/100 | train_loss=0.9122 | val_loss=0.9228 | tempo=29.4s
epoch  14/100 | train_loss=0.9080 | val_loss=0.